In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")

print("Telemetry Schema:")
telemetry.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)



In [0]:
from pyspark.sql.functions import min, max, count, countDistinct

telemetry.select(
    min("datetime").alias("start_time"),
    max("datetime").alias("end_time"),
    count("*").alias("total_records"),
    countDistinct("machineID").alias("total_machines")
).show()

+-------------------+-------------------+-------------+--------------+
|         start_time|           end_time|total_records|total_machines|
+-------------------+-------------------+-------------+--------------+
|2015-01-01 06:00:00|2016-01-01 06:00:00|       876100|           100|
+-------------------+-------------------+-------------+--------------+



In [0]:
from pyspark.sql.functions import col, hour, minute

telemetry.select(
    "datetime",
    "machineID",
    "volt",
    "vibration"
).orderBy("machineID", "datetime").show(20, truncate=False)

+-------------------+---------+----------------+----------------+
|datetime           |machineID|volt            |vibration       |
+-------------------+---------+----------------+----------------+
|2015-01-01 06:00:00|1        |176.217853015625|45.0876857639276|
|2015-01-01 07:00:00|1        |162.87922289706 |43.4139726834815|
|2015-01-01 08:00:00|1        |170.989902405567|34.1788471214451|
|2015-01-01 09:00:00|1        |162.462833264092|41.1221440884256|
|2015-01-01 10:00:00|1        |157.61002119306 |25.9905109982024|
|2015-01-01 11:00:00|1        |172.504839196295|35.6550173268837|
|2015-01-01 12:00:00|1        |156.556030606329|42.7539196974773|
|2015-01-01 13:00:00|1        |172.522780814836|35.4820086610704|
|2015-01-01 14:00:00|1        |175.324523915223|45.4822868466294|
|2015-01-01 15:00:00|1        |169.218423246933|39.9017354356787|
|2015-01-01 16:00:00|1        |167.060980719256|42.6757999060571|
|2015-01-01 17:00:00|1        |160.263953725914|38.5436809273919|
|2015-01-0

In [0]:
from pyspark.sql.functions import col, sum

telemetry.select(
    sum(col("datetime").isNull().cast("int")).alias("null_datetime"),
    sum(col("machineID").isNull().cast("int")).alias("null_machineID"),
    sum(col("volt").isNull().cast("int")).alias("null_voltage"),
    sum(col("vibration").isNull().cast("int")).alias("null_vibration")
).show()

+-------------+--------------+------------+--------------+
|null_datetime|null_machineID|null_voltage|null_vibration|
+-------------+--------------+------------+--------------+
|            0|             0|           0|             0|
+-------------+--------------+------------+--------------+



In [0]:
from pyspark.sql.functions import col

lstm_data = (
    telemetry
    .select(
        "datetime",
        "machineID",
        "volt",
        "vibration"
    )
    .orderBy("machineID", "datetime")
)

print("LSTM data schema:")
lstm_data.printSchema()

lstm_data.show(10, truncate=False)

LSTM data schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- vibration: double (nullable = true)

+-------------------+---------+----------------+----------------+
|datetime           |machineID|volt            |vibration       |
+-------------------+---------+----------------+----------------+
|2015-01-01 06:00:00|1        |176.217853015625|45.0876857639276|
|2015-01-01 07:00:00|1        |162.87922289706 |43.4139726834815|
|2015-01-01 08:00:00|1        |170.989902405567|34.1788471214451|
|2015-01-01 09:00:00|1        |162.462833264092|41.1221440884256|
|2015-01-01 10:00:00|1        |157.61002119306 |25.9905109982024|
|2015-01-01 11:00:00|1        |172.504839196295|35.6550173268837|
|2015-01-01 12:00:00|1        |156.556030606329|42.7539196974773|
|2015-01-01 13:00:00|1        |172.522780814836|35.4820086610704|
|2015-01-01 14:00:00|1        |175.324523915223|45.4822868466294|
|2015-01-01 15:00:00|1

In [0]:
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.9/281.9 MB 192.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 125.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 129.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [tensorflow]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")

print("Telemetry Schema:")
telemetry.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)



In [0]:
telemetry_pd = (
    telemetry
    .select("datetime", "machineID", "volt", "vibration")
    .orderBy("machineID", "datetime")
    .toPandas()
)

print("Rows:", len(telemetry_pd))
print("Machines:", telemetry_pd["machineID"].nunique())

telemetry_pd.head()

Rows: 876100
Machines: 100


,datetime,machineID,volt,vibration
0,2015-01-01 06:00:00,1,176.217853,45.087686
1,2015-01-01 07:00:00,1,162.879223,43.413973
2,2015-01-01 08:00:00,1,170.989902,34.178847
3,2015-01-01 09:00:00,1,162.462833,41.122144
4,2015-01-01 10:00:00,1,157.610021,25.990511


In [0]:
from sklearn.preprocessing import MinMaxScaler

voltage_scaler = MinMaxScaler()
vibration_scaler = MinMaxScaler()

telemetry_pd["volt_scaled"] = voltage_scaler.fit_transform(
    telemetry_pd[["volt"]]
)

telemetry_pd["vibration_scaled"] = vibration_scaler.fit_transform(
    telemetry_pd[["vibration"]]
)

print("Scaling completed")

print(
    telemetry_pd[
        ["volt", "volt_scaled", "vibration", "vibration_scaled"]
    ].head(10)
)

Scaling completed
         volt  volt_scaled  vibration  vibration_scaled
0  176.217853     0.499928  45.087686          0.487945
1  162.879223     0.415395  43.413973          0.460912
2  170.989902     0.466796  34.178847          0.311752
3  162.462833     0.412756  41.122144          0.423896
4  157.610021     0.382001  25.990511          0.179498
5  172.504839     0.476397  35.655017          0.335594
6  156.556031     0.375322  42.753920          0.450251
7  172.522781     0.476511  35.482009          0.332800
8  175.324524     0.494267  45.482287          0.494318
9  169.218423     0.455570  39.901735          0.404184


In [0]:
import numpy as np

INPUT_HOURS = 24
FORECAST_HOURS = 12

X = []
y = []

for machine_id, group in telemetry_pd.groupby("machineID"):

    values = group[
        ["volt_scaled", "vibration_scaled"]
    ].values

    for i in range(
        len(values) - INPUT_HOURS - FORECAST_HOURS + 1
    ):
        X.append(
            values[i:i + INPUT_HOURS]
        )

        y.append(
            values[
                i + INPUT_HOURS:
                i + INPUT_HOURS + FORECAST_HOURS
            ]
        )

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (872600, 24, 2)
Target shape: (872600, 12, 2)


In [0]:
split_index = int(len(X) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Training input:", X_train.shape)
print("Training target:", y_train.shape)
print("Testing input:", X_test.shape)
print("Testing target:", y_test.shape)

Training input: (698080, 24, 2)
Training target: (698080, 12, 2)
Testing input: (174520, 24, 2)
Testing target: (174520, 12, 2)


In [0]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

model = Sequential([
    LSTM(64, input_shape=(24, 2)),
    RepeatVector(12),
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(2))
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f5b2da33-1391-4a17-b3df-d64b8bcc1454/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 12, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 12, 2)          │           130 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,306 (196.51 KB)

 Trainable params: 50,306 (196.51 KB)

 Non-trainable params: 0 (0.00 B)

In [0]:
history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

Epoch 1/3


W0000 00:00:1790584393.171093      67 cpu_allocator_impl.cc:82] Allocation of 120628224 exceeds 10% of free system memory.
W0000 00:00:1790584393.244931      67 cpu_allocator_impl.cc:82] Allocation of 60314112 exceeds 10% of free system memory.


2455/2455 ━━━━━━━━━━━━━━━━━━━━ 131s 53ms/step - loss: 0.0089 - val_loss: 0.0082
Epoch 2/3
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 129s 53ms/step - loss: 0.0082 - val_loss: 0.0081
Epoch 3/3
2455/2455 ━━━━━━━━━━━━━━━━━━━━ 131s 53ms/step - loss: 0.0081 - val_loss: 0.0080


In [0]:
test_loss = model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

print("Test MSE:", test_loss)

  7/682 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - loss: 0.0076

W0000 00:00:1790585083.280749      67 cpu_allocator_impl.cc:82] Allocation of 33507840 exceeds 10% of free system memory.


682/682 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - loss: 0.0081
Test MSE: 0.008131393231451511


In [0]:
# Select one machine for demonstration
machine_id = 1

machine_data = (
    telemetry_pd[
        telemetry_pd["machineID"] == machine_id
    ]
    .sort_values("datetime")
)

# Take the latest 24 hours
latest_24 = machine_data[
    ["volt_scaled", "vibration_scaled"]
].tail(INPUT_HOURS).values

# Reshape for LSTM
latest_24 = np.expand_dims(
    latest_24.astype(np.float32),
    axis=0
)

# Generate 12-hour forecast
forecast_scaled = model.predict(
    latest_24,
    verbose=0
)

# Convert back to original values
forecast_voltage = voltage_scaler.inverse_transform(
    forecast_scaled[0, :, 0].reshape(-1, 1)
).flatten()

forecast_vibration = vibration_scaler.inverse_transform(
    forecast_scaled[0, :, 1].reshape(-1, 1)
).flatten()

print("Machine:", machine_id)
print("Forecasted voltage:")
print(forecast_voltage)

print("\nForecasted vibration:")
print(forecast_vibration)

Machine: 1
Forecasted voltage:
[170.05972 170.20436 170.21596 170.13803 170.19518 170.25272 170.28276
 170.28836 170.28003 170.26611 170.25156 170.23859]

Forecasted vibration:
[40.160973 40.188255 40.20058  40.180702 40.20951  40.24043  40.26357
 40.279194 40.28989  40.29761  40.303528 40.308205]


In [0]:
from datetime import timedelta
import pandas as pd

last_datetime = machine_data["datetime"].max()

forecast_dates = [
    last_datetime + timedelta(hours=i)
    for i in range(1, FORECAST_HOURS + 1)
]

forecast_df = pd.DataFrame({
    "machineID": machine_id,
    "forecast_datetime": forecast_dates,
    "predicted_voltage": forecast_voltage,
    "predicted_vibration": forecast_vibration
})

print("12-hour forecast:")
print(forecast_df)

12-hour forecast:
    machineID   forecast_datetime  predicted_voltage  predicted_vibration
0           1 2016-01-01 07:00:00         170.059723            40.160973
1           1 2016-01-01 08:00:00         170.204361            40.188255
2           1 2016-01-01 09:00:00         170.215958            40.200581
3           1 2016-01-01 10:00:00         170.138031            40.180702
4           1 2016-01-01 11:00:00         170.195175            40.209511
5           1 2016-01-01 12:00:00         170.252716            40.240429
6           1 2016-01-01 13:00:00         170.282761            40.263569
7           1 2016-01-01 14:00:00         170.288361            40.279194
8           1 2016-01-01 15:00:00         170.280029            40.289890
9           1 2016-01-01 16:00:00         170.266113            40.297611
10          1 2016-01-01 17:00:00         170.251556            40.303528
11          1 2016-01-01 18:00:00         170.238586            40.308205


In [0]:
# Derive a monitoring threshold from historical vibration data
vibration_threshold = telemetry_pd["vibration"].quantile(0.99)

print("Historical 99th percentile vibration threshold:")
print(vibration_threshold)

# Check the 12-hour forecast against the threshold
forecast_df["vibration_alert"] = (
    forecast_df["predicted_vibration"] > vibration_threshold
)

print("\nForecast alert results:")
print(
    forecast_df[
        [
            "forecast_datetime",
            "predicted_vibration",
            "vibration_alert"
        ]
    ]
)

Historical 99th percentile vibration threshold:
54.24062779393708

Forecast alert results:
     forecast_datetime  predicted_vibration  vibration_alert
0  2016-01-01 07:00:00            40.160973            False
1  2016-01-01 08:00:00            40.188255            False
2  2016-01-01 09:00:00            40.200581            False
3  2016-01-01 10:00:00            40.180702            False
4  2016-01-01 11:00:00            40.209511            False
5  2016-01-01 12:00:00            40.240429            False
6  2016-01-01 13:00:00            40.263569            False
7  2016-01-01 14:00:00            40.279194            False
8  2016-01-01 15:00:00            40.289890            False
9  2016-01-01 16:00:00            40.297611            False
10 2016-01-01 17:00:00            40.303528            False
11 2016-01-01 18:00:00            40.308205            False


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Convert forecast pandas DataFrame to Spark DataFrame
gold_q4 = spark.createDataFrame(forecast_df)

# Add the monitoring threshold
gold_q4 = gold_q4.withColumn(
    "vibration_threshold",
    F.lit(float(vibration_threshold))
)

print("Gold Q4 schema before saving:")
gold_q4.printSchema()

target_table = "workspace.default.gold_q4_telemetry_forecast"

# Create empty Delta table only if it does not exist
if not spark.catalog.tableExists(target_table):
    gold_q4.limit(0).write \
        .format("delta") \
        .saveAsTable(target_table)

    print("Gold Q4 table created")

# MERGE into Gold table
target = DeltaTable.forName(
    spark,
    target_table
)

target.alias("t").merge(
    gold_q4.alias("s"),
    """
    t.machineID = s.machineID
    AND t.forecast_datetime = s.forecast_datetime
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Q4 telemetry forecast table updated")

print("Gold Q4 schema:")
target.toDF().printSchema()

Gold Q4 schema before saving:
root
 |-- machineID: long (nullable = true)
 |-- forecast_datetime: timestamp (nullable = true)
 |-- predicted_voltage: float (nullable = true)
 |-- predicted_vibration: float (nullable = true)
 |-- vibration_alert: boolean (nullable = true)
 |-- vibration_threshold: double (nullable = false)

Gold Q4 table created
Gold Q4 telemetry forecast table updated
Gold Q4 schema:
root
 |-- machineID: long (nullable = true)
 |-- forecast_datetime: timestamp (nullable = true)
 |-- predicted_voltage: float (nullable = true)
 |-- predicted_vibration: float (nullable = true)
 |-- vibration_alert: boolean (nullable = true)
 |-- vibration_threshold: double (nullable = true)



In [0]:
gold_q4_check = spark.table(
    "workspace.default.gold_q4_telemetry_forecast"
)

print("Gold Q4 table:")
gold_q4_check.show(20, truncate=False)

print("Gold Q4 row count:")
print(gold_q4_check.count())

Gold Q4 table:
+---------+-------------------+-----------------+-------------------+---------------+-------------------+
|machineID|forecast_datetime  |predicted_voltage|predicted_vibration|vibration_alert|vibration_threshold|
+---------+-------------------+-----------------+-------------------+---------------+-------------------+
|1        |2016-01-01 08:00:00|170.20436        |40.188255          |false          |54.24062779393708  |
|1        |2016-01-01 09:00:00|170.21596        |40.20058           |false          |54.24062779393708  |
|1        |2016-01-01 14:00:00|170.28836        |40.279194          |false          |54.24062779393708  |
|1        |2016-01-01 15:00:00|170.28003        |40.28989           |false          |54.24062779393708  |
|1        |2016-01-01 17:00:00|170.25156        |40.303528          |false          |54.24062779393708  |
|1        |2016-01-01 18:00:00|170.23859        |40.308205          |false          |54.24062779393708  |
|1        |2016-01-01 11:00:00|

In [0]:
display(gold_q4_check)

machineID,forecast_datetime,predicted_voltage,predicted_vibration,vibration_alert,vibration_threshold
1,2016-01-01T08:00:00.000Z,170.20436,40.188255,false,54.24062779393708
1,2016-01-01T09:00:00.000Z,170.21596,40.20058,false,54.24062779393708
1,2016-01-01T14:00:00.000Z,170.28836,40.279194,false,54.24062779393708
1,2016-01-01T15:00:00.000Z,170.28003,40.28989,false,54.24062779393708
1,2016-01-01T17:00:00.000Z,170.25156,40.303528,false,54.24062779393708
1,2016-01-01T18:00:00.000Z,170.23859,40.308205,false,54.24062779393708
1,2016-01-01T11:00:00.000Z,170.19518,40.20951,false,54.24062779393708
1,2016-01-01T12:00:00.000Z,170.25272,40.24043,false,54.24062779393708
1,2016-01-01T07:00:00.000Z,170.05972,40.160973,false,54.24062779393708
1,2016-01-01T10:00:00.000Z,170.13803,40.180702,false,54.24062779393708


Databricks visualization. Run in Databricks to view.

In [0]:
display(gold_q4_check)

machineID,forecast_datetime,predicted_voltage,predicted_vibration,vibration_alert,vibration_threshold
1,2016-01-01T08:00:00.000Z,170.20436,40.188255,false,54.24062779393708
1,2016-01-01T09:00:00.000Z,170.21596,40.20058,false,54.24062779393708
1,2016-01-01T14:00:00.000Z,170.28836,40.279194,false,54.24062779393708
1,2016-01-01T15:00:00.000Z,170.28003,40.28989,false,54.24062779393708
1,2016-01-01T17:00:00.000Z,170.25156,40.303528,false,54.24062779393708
1,2016-01-01T18:00:00.000Z,170.23859,40.308205,false,54.24062779393708
1,2016-01-01T11:00:00.000Z,170.19518,40.20951,false,54.24062779393708
1,2016-01-01T12:00:00.000Z,170.25272,40.24043,false,54.24062779393708
1,2016-01-01T07:00:00.000Z,170.05972,40.160973,false,54.24062779393708
1,2016-01-01T10:00:00.000Z,170.13803,40.180702,false,54.24062779393708


Databricks visualization. Run in Databricks to view.